In [0]:
from decimal import Decimal

data = [
    [1, "Ivanov","Ivan", None],
    [2, "Petrov","Petya", Decimal('100')],
    [3, None, "Vasya", Decimal('200')],
    [4, None, None, None],
]
schema = "id Integer, second_name String, first_name String, salary Decimal"

df = spark.createDataFrame(data, schema)
display(df)

id,second_name,first_name,salary
1,Ivanov,Ivan,null
2,Petrov,Petya,100
3,null,Vasya,200
4,null,null,null


In [0]:
from pyspark.sql.functions import col, coalesce, lit

# Если second_name null, подставится строка first_name
df01 = df.select(
    "id",
    coalesce(col("second_name"), col("first_name")).alias("name"),
    coalesce("salary", lit(0)).alias("salary"),
)
display(df01)

id,name,salary
1,Ivanov,0
2,Petrov,100
3,Vasya,200
4,null,0


In [0]:
from pyspark.sql.functions import col, coalesce, lit

# Если имя None, подставится строка "Неизвестно"
df01 = df.select(
    "id",
    coalesce(col("name"), lit("Неизвестно")).alias("name"),
    coalesce(col("salary"), lit(0)).alias("salary")
)
display(df01)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6609373256557344>, line 9
      3 # Если имя None, подставится строка "Неизвестно"
      4 df01 = df.select(
      5     "id",
      6     coalesce(col("name"), lit("Неизвестно")).alias("name"),
      7     coalesce(col("salary"), lit(0)).alias("salary")
      8 )
----> 9 display(df01)

File <command-6609373256557344>, line 4
      1 from pyspark.sql.functions import col, coalesce, lit
      3 # Если имя None, подставится строка "Неизвестно"
----> 4 df01 = df.select(
      5     "id",
      6     coalesce(col("name"), lit("Неизвестно")).alias("name"),
      7     coalesce(col("salary"), lit(0)).alias("salary")
      8 )
      9 display(df01)

File <command-6609373256557342>, line 11
      3 data = [
      4     [1, "Ivanov","Ivan", None],
      5     [2, "Petrov","Petya", Decimal('100')],
      6     [3, None, "Vasya", Dec

In [0]:
df.createOrReplaceTempView("tmp_df")

In [0]:
%sql
select
    second_name,
    ifnull(second_name, "Unknown") as ifnull_name,
    nvl(second_name, "Unknown") as nvl_name,
    nvl2(second_name, second_name, 'Unknown') as nvl2_name,
    nullif(second_name, "Petrov") as nullif_name
from tmp_df

second_name,ifnull_name,nvl_name,nvl2_name,nullif_name
Ivanov,Ivanov,Ivanov,Ivanov,Ivanov
Petrov,Petrov,Petrov,Petrov,null
null,Unknown,Unknown,Unknown,null


In [0]:
from pyspark.sql.functions import ifnull, nvl, nvl2, nullif, col, lit

df_tmp = df.select(
    "second_name",
    ifnull(col("second_name"), lit("Unknown")).alias("ifnull_name"),
    nvl(col("second_name"), lit("Unknown")).alias("nvl_name"),
    nvl2(col("second_name"), col("second_name"), lit("Unknown")).alias("nvl2_name"),
    nullif(col("second_name"), lit("Petrov")).alias("nullif_name")
)
display(df_tmp)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4710676369168921>, line 10
      1 from pyspark.sql.functions import ifnull, nvl, nvl2, nullif, col, lit
      3 df_tmp = df.select(
      4     "second_name",
      5     ifnull(col("second_name"), lit("Unknown")).alias("ifnull_name"),
   (...)      8     nullif(col("second_name"), lit("Petrov")).alias("nullif_name")
      9 )
---> 10 display(df_tmp)

File <command-4710676369168921>, line 3
      1 from pyspark.sql.functions import ifnull, nvl, nvl2, nullif, col, lit
----> 3 df_tmp = df.select(
      4     "second_name",
      5     ifnull(col("second_name"), lit("Unknown")).alias("ifnull_name"),
      6     nvl(col("second_name"), lit("Unknown")).alias("nvl_name"),
      7     nvl2(col("second_name"), col("second_name"), lit("Unknown")).alias("nvl2_name"),
      8     nullif(col("second_name"), lit("Petrov")).alias("null

In [0]:
df.na.drop().show()
df.na.drop("any").show()
df.na.drop("all", subset=['second_name', 'first_name', 'salary']).show()
df.na.drop("any", subset=['second_name']).show()

+---+------+-----+
|age|height| name|
+---+------+-----+
| 10|  80.0|Alice|
+---+------+-----+

+---+------+-----+
|age|height| name|
+---+------+-----+
| 10|  80.0|Alice|
+---+------+-----+



---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4710676369168922>, line 3
      1 df.na.drop().show()
      2 df.na.drop("any").show()
----> 3 df.na.drop("all", subset=['second_name', 'first_name', 'salary']).show()
      4 df.na.drop("any", subset=['second_name']).show()

File <command-8504786355584218>, line 2
      1 from pyspark.sql import Row
----> 2 df = spark.createDataFrame([
      3     Row(age=10, height=80.0, name="Alice"),
      4     Row(age=5, height=float("nan"), name="Bob"),
      5     Row(age=None, height=None, name="Tom"),
      6     Row(age=None, height=float("nan"), name=None),
      7 ])

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `second_name` cannot be resolved. Did you mean one of the following? [`age`, `height`, `name`]. SQLSTATE: 42703

JVM stacktrace:
org.apache.spark.sql.Analys

In [0]:
from pyspark.sql import Row
df = spark.createDataFrame([
    Row(age=10, height=80.0, name="Alice"),
    Row(age=5, height=float("nan"), name="Bob"),
    Row(age=None, height=None, name="Tom"),
    Row(age=None, height=float("nan"), name=None),
])



In [0]:
#DataFrame.dropna(how='any', thresh=None, subset=None)

print("Example 1: Drop the row if it contains any null or NaN.")
df.na.drop().show()

print("Example 2: Drop the row only if all its values are null or NaN.")
df.na.drop(how = "all").show()

print("Example 3: Drop rows that have less than thresh non-null and non-NaN values.")
# .drop(thresh=2) — указывает Spark оставить строку только в том случае, если в ней есть как минимум 2 заполненных столбца. Если в 
# строке только один заполненный столбец (или все пустые), она будет удалена.
# thresh имеет выше приоритет, чем how
df.na.drop(thresh=2).show()

print("Example 4: Drop rows with null and NaN values in the specified columns.")
df.na.drop(subset=['age', 'name']).show()


Example 1: Drop the row if it contains any null or NaN.
+---+------+-----+
|age|height| name|
+---+------+-----+
| 10|  80.0|Alice|
+---+------+-----+

Example 2: Drop the row only if all its values are null or NaN.
+----+------+-----+
| age|height| name|
+----+------+-----+
|  10|  80.0|Alice|
|   5|   NaN|  Bob|
|NULL|  NULL|  Tom|
+----+------+-----+

Example 3: Drop rows that have less than thresh non-null and non-NaN values.
+---+------+-----+
|age|height| name|
+---+------+-----+
| 10|  80.0|Alice|
|  5|   NaN|  Bob|
+---+------+-----+

Example 4: Drop rows with null and NaN values in the specified columns.
+---+------+-----+
|age|height| name|
+---+------+-----+
| 10|  80.0|Alice|
|  5|   NaN|  Bob|
+---+------+-----+



In [0]:
df_tmp3 = df.na.fill("Unknown").na.fill(0)
display(df_tmp3)
df_tmp3 = df.na.fill({"second_name": "****", "salary": 0})
display(df_tmp3)
df_tmp3 = df.na.replace(["Petrov"], ["Pupkin"], "second_name")
display(df_tmp3)




id,second_name,first_name,salary
1,Ivanov,Ivan,0
2,Petrov,Petya,100
3,Unknown,Vasya,200
4,Unknown,Unknown,0


id,second_name,first_name,salary
1,Ivanov,Ivan,0
2,Petrov,Petya,100
3,****,Vasya,200
4,****,null,0


id,second_name,first_name,salary
1,Ivanov,Ivan,null
2,Pupkin,Petya,100
3,null,Vasya,200
4,null,null,null
